# ?????? ????????? ?????? YOLO ?? ???????? ?????

??????? ??????????? ???????? ? Jupyter/VS Code. ?? ?????????? `best.pt` ????? ? ??????? ??????? ? ????????? ?????? ?? ??????????? ????????????. ????????? ?????????? ????????? ? ?????????: ??????????? ?????????? ? ?????? 256 px, ??????? ?? ????????? ??????? 640 px ? ????? 480 px, ????? ???????? ????????????? ? 640?512 px.

## ????? ??? ????????? ??????

???? `best.pt` ?? ??????? ??? ???????? ?? **????** ???????? ??????????? ????????????, ? ??? ????? ?? ??????? train/val/test. ??????? ?????? ?? ???????? Kaggle `train_images` ???? ???????? **??????????? ?? ??? ???????? ??????**, ?? ?? ??????????? ?????? ?????????. ????? ???????? ??????? ????????? ???????, ??????? ????????? ??????????? ???????, ??????? ?? ?????????????? ?? ? ????? ????? ???????? ? ????????? ??????. ???? ????? ??????? ???? ???, ????? ????????? ??????? ?? ???????? ??????, ?? ??????????? ????????? ??? ??????????? test ??????.

??? ??????? ??????? ???????? ??????? ????????? ?????, ?????????? `train.csv` ? `train_images/`. ??? ?????? ??????????? ??????? ???????? ?? ?????; ??? ??????????????? ?????? ? ???????? ??????? Severstal.

In [ ]:
%pip -q install ultralytics opencv-python-headless pyyaml pandas

In [ ]:
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from ultralytics import YOLO

# VS Code/Jupyter ?????? ????????? kernel ?? ????? ???????.
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'best.pt').is_file():
    project_candidates = [parent for parent in PROJECT_DIR.parents if (parent / 'best.pt').is_file()]
    if project_candidates:
        PROJECT_DIR = project_candidates[0]

CHECKPOINT = PROJECT_DIR / 'best.pt'
DATA_ROOT = PROJECT_DIR
WORK = PROJECT_DIR / 'final_model_evaluation'

TILE_WIDTH = 640
TILE_STEP = 480
TARGET_HEIGHT = 256
MODEL_HEIGHT = 512
CLASS_NAMES = {0: '?????? 1', 1: '?????? 2', 2: '?????? 3', 3: '?????? 4'}

if not CHECKPOINT.is_file():
    raise FileNotFoundError(
        f'?? ?????? {CHECKPOINT}. ???????? notebook ?? ????? ???????, ??? ????? best.pt.'
    )


def find_dataset(root):
    csv_candidates = list(root.rglob('train.csv'))
    image_dirs = [path for path in root.rglob('train_images') if path.is_dir()]
    if csv_candidates and image_dirs:
        return csv_candidates[0], image_dirs[0]
    return None


found = find_dataset(DATA_ROOT)
if found is None:
    import tkinter as tk
    from tkinter import filedialog

    dialog = tk.Tk()
    dialog.withdraw()
    dialog.attributes('-topmost', True)
    selected = filedialog.askdirectory(
        initialdir=str(PROJECT_DIR),
        title='???????? ????? ? train.csv ? train_images'
    )
    dialog.destroy()
    if not selected:
        raise RuntimeError('????? ? ???????????? ??????? ?? ???????.')
    DATA_ROOT = Path(selected)
    found = find_dataset(DATA_ROOT)
    if found is None:
        raise FileNotFoundError(
            f'? ????????? ????? {DATA_ROOT} ?? ??????? train.csv ? train_images.'
        )

CSV_PATH, IMAGE_DIR = found
WORK.mkdir(parents=True, exist_ok=True)
print('????????:', CHECKPOINT)
print('????????:', CSV_PATH)
print('???????????:', IMAGE_DIR)
print('????????????? ????? ? ???????:', WORK)

## Загрузка разметки и подготовка фрагментов

Из RLE-масок формируются ограничивающие рамки связных компонент, как в обучающем ноутбуке. Все изображения в указанной папке входят в оценку; для честного test поместите сюда только отдельную holdout-выборку.

In [ ]:
raw = pd.read_csv(CSV_PATH)
if {'ImageId', 'ClassId', 'EncodedPixels'} <= set(raw.columns):
    annotations = raw[['ImageId', 'ClassId', 'EncodedPixels']].copy()
elif {'ImageId_ClassId', 'EncodedPixels'} <= set(raw.columns):
    parts = raw['ImageId_ClassId'].astype(str).str.rsplit('_', n=1, expand=True)
    annotations = pd.DataFrame({
        'ImageId': parts[0], 'ClassId': parts[1], 'EncodedPixels': raw['EncodedPixels']
    })
else:
    raise ValueError(f'Неизвестный формат train.csv: {list(raw.columns)}')

annotations['ClassId'] = annotations['ClassId'].astype(int)
if not set(annotations['ClassId'].unique()) <= {1, 2, 3, 4}:
    raise ValueError('ClassId должен содержать только значения 1–4')
annotations['EncodedPixels'] = annotations['EncodedPixels'].fillna('').astype(str).str.strip()
annotations = annotations[annotations['EncodedPixels'].ne('')]
by_image = {name: group for name, group in annotations.groupby('ImageId')}
image_paths = sorted(IMAGE_DIR.glob('*.jpg'))
if not image_paths:
    image_paths = sorted(IMAGE_DIR.glob('*.JPG'))
if not image_paths:
    raise FileNotFoundError(f'В {IMAGE_DIR} нет JPG-изображений')

def tile_starts(width):
    if width <= TILE_WIDTH:
        return [0]
    starts = list(range(0, width - TILE_WIDTH + 1, TILE_STEP))
    last = width - TILE_WIDTH
    if starts[-1] != last:
        starts.append(last)
    return starts

def decode_rle(rle, height, width):
    flat = np.zeros(height * width, dtype=np.uint8)
    values = np.fromstring(str(rle), sep=' ', dtype=np.int64)
    if len(values) % 2:
        raise ValueError('Некорректная RLE-разметка: нечётное число значений')
    for start, length in values.reshape(-1, 2):
        begin = int(start) - 1
        end = begin + int(length)
        if begin < 0 or end > flat.size:
            raise ValueError('RLE выходит за границы изображения')
        flat[begin:end] = 1
    return flat.reshape((height, width), order='F')

def mask_boxes(mask, min_area=10):
    count, _, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    return [tuple(map(int, stats[i, :4])) for i in range(1, count)
            if int(stats[i, cv2.CC_STAT_AREA]) >= min_area]

def image_masks(path, height, width):
    group = by_image.get(path.name)
    if group is None:
        return {}
    masks = {}
    for row in group.itertuples(index=False):
        class_id = int(row.ClassId) - 1
        mask = decode_rle(row.EncodedPixels, height, width)
        masks[class_id] = np.maximum(masks[class_id], mask) if class_id in masks else mask
    return masks

def labels_for_tile(masks, start, actual_width):
    lines = []
    for class_id, mask in masks.items():
        for x, y, box_width, box_height in mask_boxes(mask[:, start:start + actual_width]):
            xc = (x + box_width / 2) / TILE_WIDTH
            yc = (y + box_height / 2) / TARGET_HEIGHT
            lines.append(f'{class_id} {xc:.6f} {yc:.6f} {box_width / TILE_WIDTH:.6f} {box_height / TARGET_HEIGHT:.6f}')
    return lines

EVAL_IMAGES = WORK / 'tiles' / 'images' / 'eval'
EVAL_LABELS = WORK / 'tiles' / 'labels' / 'eval'
EVAL_IMAGES.mkdir(parents=True, exist_ok=True)
EVAL_LABELS.mkdir(parents=True, exist_ok=True)
tile_count = 0
positive_tiles = 0
class_image_counts = Counter()

for index, source in enumerate(image_paths, start=1):
    image = cv2.imread(str(source))
    if image is None:
        raise ValueError(f'Не удалось прочитать изображение: {source}')
    height, width = image.shape[:2]
    if height != TARGET_HEIGHT:
        raise ValueError(f'{source.name}: высота {height}, ожидалась {TARGET_HEIGHT}')
    masks = image_masks(source, height, width)
    for class_id in masks:
        class_image_counts[class_id] += 1
    for start in tile_starts(width):
        stem = f'{source.stem}_{start:04d}'
        crop = np.zeros((TARGET_HEIGHT, TILE_WIDTH, 3), dtype=np.uint8)
        piece = image[:, start:start + TILE_WIDTH]
        crop[:, :piece.shape[1]] = piece
        model_tile = cv2.resize(crop, (TILE_WIDTH, MODEL_HEIGHT), interpolation=cv2.INTER_LINEAR)
        image_file = EVAL_IMAGES / f'{stem}.jpg'
        if not cv2.imwrite(str(image_file), model_tile, [cv2.IMWRITE_JPEG_QUALITY, 94]):
            raise OSError(f'Не удалось записать фрагмент: {image_file}')
        lines = labels_for_tile(masks, start, min(TILE_WIDTH, width - start))
        (EVAL_LABELS / f'{stem}.txt').write_text('\n'.join(lines), encoding='utf-8')
        tile_count += 1
        positive_tiles += bool(lines)
    if index % 2000 == 0:
        print(f'Подготовлено изображений: {index} / {len(image_paths)}')

DATA_YAML = WORK / 'steel_eval.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(WORK / 'tiles'), 'val': 'images/eval', 'names': CLASS_NAMES
}, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(f'Готово: {len(image_paths)} изображений, {tile_count} фрагментов, {positive_tiles} фрагментов с рамками')
display(pd.DataFrame([
    {'Класс': CLASS_NAMES[i], 'Изображений с классом': class_image_counts[i]}
    for i in range(4)
]))

## Оценка финального чекпойнта

Ultralytics считает Precision/Recall и AP по рамкам при IoU-порогах 0.50 и 0.50:0.95. Эти метрики относятся к фрагментам 640×512, как и валидация при обучении.

In [ ]:
model = YOLO(str(CHECKPOINT))
metrics = model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=640,
    batch=16,
    conf=0.001,
    iou=0.7,
    plots=True,
    project=str(WORK / 'runs'),
    name='final_checkpoint_eval',
    exist_ok=True,
)

summary = pd.DataFrame([{
    'Изображений': len(image_paths),
    'Фрагментов': tile_count,
    'Precision': float(metrics.box.mp),
    'Recall': float(metrics.box.mr),
    'mAP50': float(metrics.box.map50),
    'mAP50-95': float(metrics.box.map),
}])
per_class = pd.DataFrame({
    'ClassId': metrics.box.ap_class_index,
    'Класс': [CLASS_NAMES[int(i)] for i in metrics.box.ap_class_index],
    'AP50': metrics.box.ap50,
    'AP50-95': metrics.box.ap,
})
display(summary)
display(per_class)

results_dir = WORK / 'runs' / 'final_checkpoint_eval'
summary.to_csv(results_dir / 'final_metrics.csv', index=False)
per_class.to_csv(results_dir / 'final_metrics_per_class.csv', index=False)
print('Графики и таблицы сохранены в:', results_dir)

## ?????????????

- ???? ??????????? ??????????? ?? ????????? Kaggle `train_images`, ??????? ? ??????: **??????????????? ??????? ?? ??????, ?????????????? ??? ????????**. ?? ????????? ?? ???????????? test-?????????.
- ??? ??????????? ?????? ??????????? ????????? ??????????? ???????, ?? ?????????????? ??? ????????, ??????? ?????? ??? ???? ???????; ??? ??????? ???????? ?? ?????.
- `mAP50-95` ?????? `mAP50` ? ??????? ???????? ???????? ???????????. ???????? ????? AP ??????? ??????: ?????? ????? 2 ????? ????? ??????? ????? ?????????? ??????.
- ????????? CSV ? ??????? ????????? ? ???????? `final_model_evaluation/runs/final_checkpoint_eval` ????? ? ??????? ???????.